# **Offer Recommender System for Telecom Products/Services**


## **Introduction**

Offer recommendation in the telecom industry refers to the process of **suggesting personalized offers** to customers based on their usage patterns, demographics, and other relevant factors. Offering recommendations in the telecom industry is crucial for improving customer satisfaction and increasing revenue. By using advanced algorithms and big data techniques, telecom companies can provide customers with relevant and valuable offers, leading to higher customer loyalty and long-term growth.

### **Business Impact of Offer Recommendation**

**Increase customer satisfaction**: Offer recommendation allows telecom companies to understand their customers better and provide them with relevant and valuable offers, leading to higher customer satisfaction.


**Increase revenue**: By providing customers with relevant and valuable offers, telecom companies can increase their revenue through increased customer spending and reduced churn.


**Improve customer loyalty**: By providing personalized offers to customers, telecom companies can create a better customer experience and build stronger relationships with their customers, leading to higher customer loyalty.


## **Possible solutions**


There are various ways to build an offer recommendation system, and the choice depends on factors such as the data available, the complexity of the system, and the computational resources available. Some approaches are:


### **1. Rule-based systems:**
 In this approach, offers are recommended based on predefined rules. Rule-based systems are easy to implement and interpret, but they may not be very accurate or adaptive. It makes sense to use them when you have a specific business mandate (example - we want to use product A to promote product B because of a Marketing Strategy so we will recommend product B to everyone who bot A).


### **2. Collaborative filtering:**
 Collaborative filtering is a type of recommendation that recommends offers based on the preferences of similar customers. In this approach, we find what customers are similar among themselves and similar offers to similar customers.


### **3. Content-based filtering:**
In this approach, the system builds a model of the customer's preferences based on the features of the offers they have interacted with and then recommends offers that are similar in terms of those features. Content-based filtering is useful when the system has access to rich feature data, but it can struggle with the cold start problem (i.e., recommending new offers to customers who have not interacted with any offers yet). This is the type of thing that powers retailers' recommendations - they get similar products to what you bought and recommend it to you.


### **4. Hybrid systems:**
Hybrid systems combine different approaches to leverage their strengths and mitigate their weaknesses. For example, a system might use a collaborative filtering approach to recommend offers to customers who have already interacted with offers, and a content-based approach to recommend offers to new customers.




## **Approach**


We are treating this problem as an unsupervised learning problem. This means that, in practice, we don't have a way in this dataset to validate if what we did is 'right' or 'wrong'.


In real life, the approach here would be to test this algorithm with real customers to see if this improves churn.


Given our assumptions about the offers, we will build a collaborative-filtering system based on the user. Simplifying, here's the logic of what we'll build:


1. We'll build an algorithm to identify who identifies, for customer A, who are the n-most similar customers;
2. We'll use a churn-rate approach to identify, among the similar customers, what is the most successful offer;
3. We will then choose the most successful offer to provide to our customer A.


We'll be training this algorithm on part of the dataset of customers who have received the offer (have A,B,C,D,E,F,G,H,I or J) in their 'offer' field. The idea is to apply that to the 'No Offer' group. 



## **Learning Outcomes**


* How to load data from AWS SQL using pyodbc and pandas
* Exploratory Data Analysis
* Categorical Feature Encoding using Labelencoder
* Data Preprocessing and Feature Engineering
* Understanding Cosine Similarity, Manhattan distance, Euclidean distance
* How to choose a distance metric for a specific problem?
* What is the minimum threshold parameter?
* Build a user based collaborative offer recommendation system
* Bootstrapping the offer recommendation system
* Testing of the algorithm in production


## **Package Requirements**

In [43]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances

## **The data**

### **AWS S3 - CSV**
Use the S3 public link to read the CSV file directly into a pandas DataFrame


### **The database**

In [44]:
# use s3_link if you are connected with internet
s3_link='https://s3.amazonaws.com/projex.dezyre.com/recommender-system-for-telecom-products/materials/Telecom_data.csv'
df = pd.read_csv(s3_link)

## if not use following and provide the path accordingly
# csv_file_path='../data/Telecom_data.csv'
# df = pd.read_csv(csv_file_path)

In [45]:
df.head(5)

,Customer ID,Month,Month of Joining,zip_code,Gender,Age,Married,Dependents,Number of Dependents,Location ID,...,Streaming Movies,Streaming Music,Unlimited Data,Payment Method,Status ID,Satisfaction Score,Churn Category,Churn Reason,Customer Status,Churn Value
0,hthjctifkiudi0,1,1,71638,Female,36.000000,No,No,0.0,jeavwsrtakgq0,...,Yes,Yes,Yes,Credit Card,vvhwtmkbxtvsppd52013,3,Competitor,Competitor offered higher download speeds,Churned,1
1,uqdtniwvxqzeu1,6,6,72566,Male,36.472065,No,No,0.0,qcvetdmalnkw1,...,No,No,No,Bank Withdrawal,jucxaluihiluj82863,4,Not Applicable,Not Applicable,Stayed,0
2,uqdtniwvxqzeu1,7,6,72566,Male,36.442687,No,No,0.0,qcvetdmalnkw1,...,No,No,Yes,Credit Card,vjskkxphumfai57182,3,Not Applicable,Not Applicable,Stayed,0
3,uqdtniwvxqzeu1,8,6,72566,Male,36.837888,No,No,0.0,qcvetdmalnkw1,...,No,No,Yes,Wallet Balance,cdwbcrvylqca53109,4,Not Applicable,Not Applicable,Stayed,0
4,uqdtniwvxqzeu1,9,6,72566,Male,36.490214,No,No,0.0,qcvetdmalnkw1,...,Yes,No,Yes,Credit Card,whqrmeulitfj98550,1,Not Applicable,Not Applicable,Stayed,0


In [46]:
print(df.shape)
print(df.columns)

(653753, 74)
Index(['Customer ID', 'Month', 'Month of Joining', 'zip_code', 'Gender', 'Age',
       'Married', 'Dependents', 'Number of Dependents', 'Location ID',
       'Service ID', 'state', 'county', 'timezone', 'area_codes', 'country',
       'latitude', 'longitude', 'roam_ic', 'roam_og', 'loc_og_t2t',
       'loc_og_t2m', 'loc_og_t2f', 'loc_og_t2c', 'std_og_t2t', 'std_og_t2m',
       'std_og_t2f', 'std_og_t2c', 'isd_og', 'spl_og', 'og_others',
       'loc_ic_t2t', 'loc_ic_t2m', 'loc_ic_t2f', 'std_ic_t2t', 'std_ic_t2m',
       'std_ic_t2f', 'std_ic_t2o', 'spl_ic', 'isd_ic', 'ic_others',
       'total_rech_amt', 'total_rech_data', 'vol_4g', 'vol_5g', 'arpu_5g',
       'arpu_4g', 'arpu', 'night_pck_user', 'fb_user', 'aug_vbc_5g', 'offer',
       'Referred a Friend', 'Number of Referrals', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Internet Type',
       'Streaming Data Consumption', 'Online Security', 'Online Backup',
       'Device Protection Plan', 'Premium Tec

When we inspect the dataframe, we notice the same customer appears more than once, in different months.

We have 2 options here:
1. Create a consolidated frame where each customer is in 1 line;
2. Use the different moments in time of each customer as a datapoint;

The latter approach gives us more data points in theory, because in different months customers could have behaved differently. We will follow with approach number 2. However, we need to keep this in mind as we calculate similarity scores.

## **Exploratory Data Analysis**

### **Data Dictionary**



| Column name	 | Description|
| ----- | ----- |
| Customer ID	 | Unique identifier for each customer |
| Month | Calendar Month- 1:12 | 
| Month of Joining |	Calender Month -1:14, Month for which the data is captured|
| zip_code |	Zip Code|
|Gender |	Gender|
| Age |	Age(Years)|
| Married |	Marital Status |
|Dependents | Dependents - Binary |
| Number of Dependents |	Number of Dependents|
|Location ID |	Location ID|
|Service ID	 |Service ID|
|state|	State|
|county	|County|
|timezone	|Timezone|
|area_codes|	Area Code|
|country	|Country|
|latitude|	Latitude|
|longitude	|Longitude|
|arpu|	Average revenue per user|
|roam_ic	|Roaming incoming calls in minutes|
|roam_og	|Roaming outgoing calls in minutes|
|loc_og_t2t|	Local outgoing calls within same network in minutes|
|loc_og_t2m	|Local outgoing calls outside network in minutes(outside same + partner network)|
|loc_og_t2f|	Local outgoing calls with Partner network in minutes|
|loc_og_t2c	|Local outgoing calls with Call Center in minutes|
|std_og_t2t|	STD outgoing calls within same network in minutes|
|std_og_t2m|	STD outgoing calls outside network in minutes(outside same + partner network)|
|std_og_t2f|	STD outgoing calls with Partner network in minutes|
|std_og_t2c	|STD outgoing calls with Call Center in minutes|
|isd_og|	ISD Outgoing calls|
|spl_og	|Special Outgoing calls|
|og_others|	Other Outgoing Calls|
|loc_ic_t2t|	Local incoming calls within same network in minutes|
|loc_ic_t2m|	Local incoming calls outside network in minutes(outside same + partner network)|
|loc_ic_t2f	|Local incoming calls with Partner network in minutes|
|std_ic_t2t	|STD incoming calls within same network in minutes|
|std_ic_t2m	|STD incoming calls outside network in minutes(outside same + partner network)|
|std_ic_t2f|	STD incoming calls with Partner network in minutes|
|std_ic_t2o|	STD incoming calls operators other networks in minutes|
|spl_ic|	Special Incoming calls in minutes|
|isd_ic|	ISD Incoming calls in minutes|
|ic_others|	Other Incoming Calls|
|total_rech_amt|	Total Recharge Amount in Local Currency|
|total_rech_data|	Total Recharge Amount for Data in Local Currency
|vol_4g|	4G Internet Used in GB|
|vol_5g|	5G Internet used in GB|
|arpu_5g|	Average revenue per user over 5G network|
|arpu_4g|	Average revenue per user over 4G network|
|night_pck_user|	Is Night Pack User(Specific Scheme)|
|fb_user|	Social Networking scheme|
|aug_vbc_5g|	Volume Based cost for 5G network (outside the scheme paid based on extra usage)|
|offer|	Offer Given to User|
|Referred a Friend|	Referred a Friend : Binary|
|Number of Referrals|	Number of Referrals|
|Phone Service|	Phone Service: Binary|
|Multiple Lines|	Multiple Lines for phone service: Binary|
|Internet Service|	Internet Service: Binary|
|Internet Type|	Internet Type|
|Streaming Data Consumption|	Streaming Data Consumption|
|Online Security|	Online Security|
|Online Backup|	Online Backup|
|Device Protection Plan|	Device Protection Plan|
|Premium Tech Support|	Premium Tech Support|
|Streaming TV|	Streaming TV|
|Streaming Movies|	Streaming Movies|
|Streaming Music|	Streaming Music|
|Unlimited Data|	Unlimited Data|
|Payment Method|	Payment Method|
|Status ID|	Status ID|
|Satisfaction Score|	Satisfaction Score|
|Churn Category|	Churn Category|
|Churn Reason|	Churn Reason|
|Customer Status|	Customer Status|
|Churn Value|	Binary Churn Value



In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653753 entries, 0 to 653752
Data columns (total 74 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Customer ID                 653753 non-null  object 
 1   Month                       653753 non-null  int64  
 2   Month of Joining            653753 non-null  int64  
 3   zip_code                    653753 non-null  int64  
 4   Gender                      653753 non-null  object 
 5   Age                         653753 non-null  float64
 6   Married                     653753 non-null  object 
 7   Dependents                  653753 non-null  object 
 8   Number of Dependents        653753 non-null  float64
 9   Location ID                 653753 non-null  object 
 10  Service ID                  653753 non-null  object 
 11  state                       653753 non-null  object 
 12  county                      653753 non-null  object 
 13  timezone      

In [48]:
df['offer'].unique() # how many unique offers are there

array(['A', 'F', 'No Offer', 'J', 'E', 'C', 'I', 'B', 'D', 'H', 'G'],
      dtype=object)

In [49]:
# Taking a look at the offer distribution
dfg = df.groupby('offer').agg({'Customer ID':'count'}).reset_index()
dfg['% Total'] = dfg['Customer ID']/dfg['Customer ID'].sum() #this creates a % of total column
dfg['% Total'] = dfg['% Total'].apply(lambda x: '{:.2%}'.format(x)) #this function simply formats the column to %
dfg #this displays the dataframe

,offer,Customer ID,% Total
0,A,15915,2.43%
1,B,15871,2.43%
2,C,16153,2.47%
3,D,16162,2.47%
4,E,15833,2.42%
5,F,15789,2.42%
6,G,15778,2.41%
7,H,15855,2.43%
8,I,15860,2.43%
9,J,15893,2.43%


**Observation**
* The Offers seems to be evenly distributed amongst customers
* There are about 76% users who did not receive any offer from the company


In [50]:
# Taking a look at the offer distribution and churn distribution
dfg2 = df.groupby(['offer','Customer Status']).agg({'Customer ID':'count'}).reset_index()
pivoted_dfg2 = dfg2.pivot(index='offer', columns='Customer Status', values='Customer ID')
pivoted_dfg2 = pivoted_dfg2.reset_index()
pivoted_dfg2['Churn Rate'] = pivoted_dfg2['Churned']/(pivoted_dfg2['Churned'] + pivoted_dfg2['Stayed'])
pivoted_dfg2['Churn Rate'] = pivoted_dfg2['Churn Rate'].apply(lambda x: '{:.2%}'.format(x)) #this function simply formats the column to %
pivoted_dfg2

Customer Status,offer,Churned,Stayed,Churn Rate
0,A,1464,14451,9.20%
1,B,1441,14430,9.08%
2,C,1470,14683,9.10%
3,D,1484,14678,9.18%
4,E,1418,14415,8.96%
5,F,1459,14330,9.24%
6,G,1409,14369,8.93%
7,H,1477,14378,9.32%
8,I,1468,14392,9.26%
9,J,1483,14410,9.33%


**Observations**

* churn rate seems to be similar amongst customers regardless of the offer they received -> this tells us that maybe offers are not being tailored enough to groups


In [51]:
# Taking a look at the churn category
dfg2 = df.groupby(['Churn Category',]).agg({'Customer ID':'count'}).reset_index()
dfg2['% Total'] = dfg2['Customer ID']/dfg2['Customer ID'].sum() #this creates a % of total column
dfg2['% Total'] = dfg2['% Total'].apply(lambda x: '{:.2%}'.format(x)) #this function simply formats the column to %
dfg2 #this displays the dataframe

,Churn Category,Customer ID,% Total
0,Attitude,296,0.05%
1,Competitor,5974,0.91%
2,Dissatisfaction,6001,0.92%
3,Not Applicable,622748,95.26%
4,Other,4356,0.67%
5,Price,4381,0.67%
6,Support,7538,1.15%
7,Unknown,1269,0.19%
8,bcvjhdjcb,1190,0.18%


**Observations**

* The Churn Category for Competitor, Dissatisfaction, Price, Support have higher customers
* We can give them specific offers which may lead them to stay rather than churning


## **Data Processing**

### **Missing Value Detection and Imputation**

In [52]:
percent_missing = df.isna().sum() * 100 / len(df)
print(percent_missing[percent_missing > 0])

total_rech_data    32.107539
night_pck_user     57.070943
fb_user            62.775085
Multiple Lines      7.048534
Internet Type      49.751206
Unlimited Data      1.698348
dtype: float64


In [53]:
missing_value_df = pd.DataFrame({'column_name': df.columns,
                                 'percent_missing': percent_missing.values})
missing_value_df.sort_values(by='percent_missing',ascending=False)

,column_name,percent_missing
49,fb_user,62.775085
48,night_pck_user,57.070943
57,Internet Type,49.751206
42,total_rech_data,32.107539
55,Multiple Lines,7.048534
...,...,...
24,std_og_t2t,0.000000
23,loc_og_t2c,0.000000
22,loc_og_t2f,0.000000
21,loc_og_t2m,0.000000


**Observation**

*  Columns 'fb_user' and 'night_pck_user' have more 50% missing value. We will simply **drop** this from our dataframe
* According to data dictionary 'Internet Type' and 'total_rech_data' seems to correlated.
* We need to check for columns 'Internet Type' and 'total_rech_data' and **impute missing values if possible**


In [54]:
df=df.drop(columns=['fb_user','night_pck_user'])

In [55]:
df['total_rech_data'].isna().sum()

209904

In [56]:
df['Internet Type'].isna().sum()

325250

**Observation:**

*  These missing values may represent customers who have not recharged their account or have recharged but the information has not been recorded.

* It is possible that customers with missing recharge data are those who received free data service, and therefore did not need to recharge their account. Alternatively, it is possible that the missing values are due to technical issues, such as data recording errors or system failures.

In [57]:
# Checking the value counts of Internet Service where total recharge data was null
df[df['total_rech_data'].isna()]['Internet Service'].value_counts(dropna=False)

Internet Service
Yes    209904
Name: count, dtype: int64

In [58]:
df[(df['total_rech_data'].isna())]['Unlimited Data'].value_counts()

Unlimited Data
Yes    181040
No      28864
Name: count, dtype: int64

In [59]:
df[(df['total_rech_data'].isna())][['arpu_4g','arpu_5g']].value_counts()

arpu_4g         arpu_5g       
Not Applicable  Not Applicable    195182
297.57          8530.983629            4
544.17          8536.565906            3
395.94          8533.210427            3
290.09          8530.814304            3
                                   ...  
222.42          1468.94                1
222.56          8529.28563             1
222.67          8529.28812             1
222.73          8529.289478            1
2559.56         8582.188229            1
Name: count, Length: 14247, dtype: int64

**Observation**:

* We can fill the missing values in the total_rech_data column with 0 when the arpu (Average Revenue Per User) is not applicable. This is because the arpu is a measure of the revenue generated per user, and if it is not applicable, it may indicate that the user is not generating any revenue for the company. In such cases, it is reasonable to assume that the total data recharge amount is 0
* It is advisable to check with the business before making this decision 

In [60]:
# Replacing all values of total recharge data= 0 where arpu 4g and 5g are not applicable
df.loc[(df['arpu_4g']=='Not Applicable') | (df['arpu_5g']=='Not Applicable'),'total_rech_data']=0
# Missing value percentage after imputation
df['total_rech_data'].isna().sum()/df.shape[0]

0.022519208324856637

In [61]:
# Calculate the mean of 'total_rech_data' where either 'arpu_4g' or 'arpu_5g' is not equal to 'Not Applicable'
arpu_data_mean=df.loc[(df['arpu_4g']!='Not Applicable') | (df['arpu_5g']!='Not Applicable'),'total_rech_data'].mean()
arpu_data_mean

4.85274721808543

In [62]:
# Fill NaN values in 'total_rech_data' with the mean of 'total_rech_data' where either 'arpu_4g' or 'arpu_5g' is not equal to 'Not Applicable'
df['total_rech_data']=df['total_rech_data'].fillna(arpu_data_mean)
df['total_rech_data'].isna().sum()

0

In [63]:
# Check the value counts for Internet Type
df['Internet Type'].value_counts(dropna=False)

Internet Type
NaN            325250
Fiber Optic    134991
Cable          112100
DSL             81412
Name: count, dtype: int64

In [64]:
# Check value counts for Internet Service where Internet Type is null
df[df['Internet Type'].isna()]['Internet Service'].value_counts(dropna=False)

Internet Service
No     236152
Yes     89098
Name: count, dtype: int64

In [65]:
# Filling Null values in Internet Type 
df['Internet Type']=df['Internet Type'].fillna('Not Applicable')

In [66]:
# Replace 'Not Applicable' with 0 in 'arpu_4g'
df['arpu_4g'] = df['arpu_4g'].replace('Not Applicable', 0)

# Replace 'Not Applicable' with 0 in 'arpu_5g'
df['arpu_5g'] = df['arpu_5g'].replace('Not Applicable', 0)

# Convert 'arpu_4g' to float data type
df['arpu_4g'] = df['arpu_4g'].astype(float)

# Convert 'arpu_5g' to float data type
df['arpu_5g'] = df['arpu_5g'].astype(float)

### **Outlier Detection and Imputation**


Outlier detection is a critical data analysis technique that involves identifying and removing data points that are significantly different from the rest of the data. Outliers are data points that lie far away from the rest of the data, and they can significantly influence the statistical analysis and machine learning models' performance. Therefore, identifying and removing outliers is essential to ensure accurate and reliable data analysis results.

There are two main approaches for outlier detection: parametric and non-parametric.

* Parametric Methods:
Parametric methods assume that the data follows a specific distribution, such as a normal distribution. In this approach, outliers are identified by calculating the distance of each data point from the mean of the distribution in terms of the number of standard deviations. Data points that are beyond a certain number of standard deviations (usually three or more) are considered as outliers.

One common parametric method is the Z-score method, which calculates the distance of each data point from the mean in terms of standard deviations.
Parametric methods can be useful when the data follows a known distribution, but they may not be effective when the data is not normally distributed.

* Non-Parametric Methods:
Non-parametric methods do not assume any specific distribution of the data. Instead, they rely on the rank or order of the data points. In this approach, outliers are identified by comparing the values of each data point with the values of other data points. Data points that are significantly different from other data points are considered as outliers.

Quantiles are an important concept in non-parametric outlier detection methods. They represent values that divide a dataset into equal-sized parts, such as quarters or thirds. The most commonly used quantiles are the median (which divides the data into two equal parts), the first quartile (which divides the data into the lowest 25% and the highest 75%), and the third quartile (which divides the data into the lowest 75% and the highest 25%).

The interquartile range (IQR) is another important concept related to quantiles. It is defined as the difference between the third and first quartiles and represents the middle 50% of the data. The IQR can be used to identify outliers by defining a range (known as the Tukey's fence) beyond which any data points are considered outliers.
Non-parametric methods can be useful when the data is not normally distributed or when the distribution is unknown.

In [67]:
# List of continuous columns
cts_cols=['Age','Number of Dependents',
       'roam_ic', 'roam_og', 'loc_og_t2t',
       'loc_og_t2m', 'loc_og_t2f', 'loc_og_t2c', 'std_og_t2t', 'std_og_t2m',
       'std_og_t2f', 'std_og_t2c', 'isd_og', 'spl_og', 'og_others',
       'loc_ic_t2t', 'loc_ic_t2m', 'loc_ic_t2f', 'std_ic_t2t', 'std_ic_t2m',
       'std_ic_t2f', 'std_ic_t2o', 'spl_ic', 'isd_ic', 'ic_others',
       'total_rech_amt', 'total_rech_data', 'vol_4g', 'vol_5g', 'arpu_5g',
       'arpu_4g', 'arpu', 'aug_vbc_5g', 'Number of Referrals','Satisfaction Score',
       'Streaming Data Consumption']   

# Create an empty dataframe with columns as cts_cols and index as quantiles
quantile_df=pd.DataFrame(columns=cts_cols,index=[0.1,0.25,0.5,0.75,0.8,0.9,0.95,0.97,0.99])

# for each column in cts_cols, calculate the corresponding quantiles and store them in the quantile_df
for col in cts_cols:
   quantile_df[col]=df[col].quantile([0.1,0.25,0.5,0.75,0.8,0.9,0.95,0.97,0.99])
quantile_df

,Age,Number of Dependents,roam_ic,roam_og,loc_og_t2t,loc_og_t2m,loc_og_t2f,loc_og_t2c,std_og_t2t,std_og_t2m,...,total_rech_data,vol_4g,vol_5g,arpu_5g,arpu_4g,arpu,aug_vbc_5g,Number of Referrals,Satisfaction Score,Streaming Data Consumption
0.10,24.0,0.0,0.000000,0.000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0000,0.0000,0.000000,0.000000,-256.2000,0.0000,0.0,1.0,0.0
0.25,28.0,0.0,12.090000,14.710,32.7000,26.260000,1.460000,1.610000,33.120000,25.560000,...,0.000000,0.0000,0.0000,0.000000,0.000000,118.9400,0.0000,0.0,3.0,2.0
0.50,34.0,0.0,50.560000,75.100,171.3300,135.460000,7.800000,8.180000,174.600000,134.800000,...,0.000000,47.0100,362.3800,0.000000,0.000000,348.5400,117.3200,4.0,3.0,20.0
0.75,43.0,1.0,162.030000,135.280,309.0900,618.210000,14.090000,14.700000,316.240000,244.490000,...,2.000000,154.9000,964.7200,194.470000,228.220000,580.6500,311.7500,8.0,4.0,49.0
0.80,47.0,2.0,496.902000,146.820,856.7660,1392.937844,43.870000,15.970000,344.970000,266.540000,...,4.852747,176.3600,3814.2220,789.000000,783.290000,626.2300,350.4900,8.0,4.0,56.0
0.90,55.0,4.0,969.043806,689.598,3613.9960,2644.568000,126.593474,109.097149,1547.136000,1007.842370,...,14.000000,219.2680,12369.5160,2219.752000,2224.100000,1901.5140,789.0000,10.0,5.0,69.0
0.95,61.0,7.0,1283.198000,1954.392,5079.8300,3479.438000,183.490000,207.514000,3953.108671,3108.617986,...,23.000000,663.2040,17358.4180,8530.865147,8675.302558,5892.6180,3943.2100,11.0,5.0,77.0
0.97,64.0,8.0,1494.043200,2550.390,5806.0544,3756.444400,206.750000,277.344400,5344.123200,3848.301590,...,26.000000,1438.5100,19569.9704,8724.440600,8839.721689,7592.5688,5949.3792,11.0,5.0,80.0
0.99,74.0,9.0,1646.899600,3041.760,6191.2040,4060.298800,257.650000,311.464800,6729.403200,4875.216400,...,30.000000,4289.8496,254687.0000,254687.000000,254687.000000,8846.9584,7366.7684,11.0,5.0,83.0


**Observation**

The variables vol_5g, arpu_4g, and arpu_5g seems to have some abrupt values

In [68]:
# Checking further
df['arpu_4g'].quantile([0.75,0.8,0.9,0.95,0.97,0.99,0.999])

0.750       228.220000
0.800       783.290000
0.900      2224.100000
0.950      8675.302558
0.970      8839.721689
0.990    254687.000000
0.999    254687.000000
Name: arpu_4g, dtype: float64

In [69]:
# Calculate the proportion of rows in the DataFrame where the value in the 'arpu_4g' column is equal to 254687
df[df['arpu_4g']==254687].shape[0]/df.shape[0]

0.019651152652454366

In [70]:
# Get the value counts of 'total_rech_data' for observations where the value in the 'arpu_4g' column is equal to 254687
df[df['arpu_4g']==254687]['total_rech_data'].value_counts()

total_rech_data
0.0    12847
Name: count, dtype: int64

Now, since the recharge amount is 0 and there is no ARPU, let's replace it with 0.

In [71]:
# Replace the outlier value 254687 in the 'arpu_4g' column of the dataframe 'df' with 0.
df['arpu_4g']=df['arpu_4g'].replace(254687,0)

In [72]:
# Checking further
df['arpu_4g'].quantile([0.75,0.8,0.9,0.95,0.97,0.99,0.999])

0.750      120.570000
0.800      504.112000
0.900     1893.758000
0.950     2493.880000
0.970     8675.470757
0.990     8839.721689
0.999    87978.000000
Name: arpu_4g, dtype: float64

In [73]:
# Filter by 'arpu_4g' value of 87978 and count unique values in 'total_rech_data' column
df[df['arpu_4g']==87978]['total_rech_data'].value_counts()

total_rech_data
0.0    5007
Name: count, dtype: int64

All rows in the dataframe with an 'arpu_4g' value of 87978 have 0 value in the 'total_rech_data' column, indicating that these are likely outliers. Therefore, we have decided to replace the 'arpu_4g' value for these rows with 0.

In [74]:
# Replace the values with 0
df['arpu_4g']=df['arpu_4g'].replace(87978,0)

In [75]:
# Checking the quantiles again
df['arpu_4g'].quantile([0.75,0.8,0.9,0.95,0.97,0.99,0.999])

0.750     107.760000
0.800     432.246000
0.900    1803.560000
0.950    2424.072000
0.970    2735.554400
0.990    8705.097343
0.999    8839.721689
Name: arpu_4g, dtype: float64

This seems to be fairly good now

In [76]:
# Get the value counts of 'total_rech_data' for observations where the value in the 'arpu_5g' column is equal to 254687
df[df['arpu_5g']==254687]['total_rech_data'].value_counts()

total_rech_data
0.0    12614
Name: count, dtype: int64

In [77]:
# Get the value counts of 'total_rech_data' for observations where the value in the 'arpu_5g' column is equal to 87978
df[df['arpu_5g']==87978]['total_rech_data'].value_counts()

total_rech_data
0.0    5130
Name: count, dtype: int64

In [78]:
# Replacing the values with 0 where total recharge data is 0
df['arpu_5g']=df['arpu_5g'].replace([87978,254687],0)

In [79]:
# Check the quantiles of ARPU 5G
df['arpu_5g'].quantile([0.75,0.8,0.9,0.95,0.97,0.99,0.999])

0.750      96.490000
0.800     417.102000
0.900    1797.618000
0.950    2543.904000
0.970    2792.060000
0.990    8587.153966
0.999    8724.440600
Name: arpu_5g, dtype: float64

This seems to be fairly good now

In [80]:
# Replace the outlier values
df['vol_5g']=df['vol_5g'].replace([87978,254687],0)

In [81]:
# Check the quantiles of Volume of 5G data
df['vol_5g'].quantile([0.75,0.8,0.9,0.95,0.97,0.98,0.99,0.999])

0.750      895.8100
0.800     1654.5460
0.900     9658.3760
0.950    14517.6400
0.970    16580.3764
0.980    17551.8796
0.990    18614.5528
0.999    19746.1824
Name: vol_5g, dtype: float64

Lets store this processed data for further use.

In [82]:
df_processed = df.copy()
#df.to_csv('processed_telecom_offer_data.csv',index=False)

## **Feature engineering**

**1. Splitting the dataset into a training and production dataset:**

- Training: part of the customers who received offers which will be used to train the model
- Production: customers who did not received offers to whom we'd like to then offer something

In [83]:
# Let's split our dataframe in a training and production dataset:
def split_dataframe(data):
    train = data[data['offer']!='No Offer']
    production = data[data['offer']=='No Offer']
    return train, production

In [84]:
train, production = split_dataframe(df_processed)

In [85]:
train.shape,production.shape

((159109, 72), (494644, 72))

Here we are not dealing with traditional train test split method as we are building an unsupervised collaborative recommender system.
Now to make model learn we need to pass all the data with respect to offers.


we are creating 2 dataframes for each train and production, that have the Customer ID as a join key. This will help us manipulate features, and also trace them back to a particular customer;


In [86]:
#This help us identify the customer and the business outcomes
id_variables = ['Customer ID', 'Month','Month of Joining','offer','Churn Category',
       'Churn Reason', 'Customer Status', 'Churn Value']


#This helps us identify the different profiles of customers
selected_variables = ['Customer ID', 'Month', 'Month of Joining', 'Gender', 'Age',
                      'Married', 'Number of Dependents', 'area_codes','roam_ic', 'roam_og',
                      'loc_og_t2t','loc_og_t2m', 'loc_og_t2f', 'loc_og_t2c', 'std_og_t2t', 'std_og_t2m',
                      'std_og_t2f', 'std_og_t2c', 'isd_og', 'spl_og', 'og_others',
                      'loc_ic_t2t', 'loc_ic_t2m', 'loc_ic_t2f', 'std_ic_t2t', 'std_ic_t2m',
                      'std_ic_t2f', 'std_ic_t2o', 'spl_ic', 'isd_ic', 'ic_others',
                      'total_rech_amt', 'total_rech_data', 'vol_4g', 'vol_5g', 'arpu_5g',
                      'arpu_4g', 'arpu', 'aug_vbc_5g','Number of Referrals', 'Phone Service',
                      'Multiple Lines', 'Internet Service', 'Internet Type',
                      'Streaming Data Consumption', 'Online Security', 'Online Backup',
                      'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
                      'Streaming Movies', 'Streaming Music', 'Unlimited Data',
                      'Payment Method']

train_id=train[id_variables]
train_feat=train[selected_variables]

prod_id=production[id_variables]
prod_feat=production[selected_variables]

In the code above, what we have essentially eliminated are complex variables like latitude, longitude and timezone because they could be represented by the area_codes variable, that represents location.

**2. Converting the Month of Joining into a customer tenure**

In [87]:
train_feat['tenure'] = train_feat['Month']- train_feat['Month of Joining']
train_feat['tenure'].describe()
prod_feat['tenure'] = prod_feat['Month']- prod_feat['Month of Joining']
prod_feat['tenure'].describe()

/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/3941133429.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_feat['tenure'] = train_feat['Month']- train_feat['Month of Joining']
/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/3941133429.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prod_feat['tenure'] = prod_feat['Month']- prod_feat['Month of Joining']


count    494644.000000
mean          3.830003
std           3.018754
min           0.000000
25%           1.000000
50%           3.000000
75%           6.000000
max          13.000000
Name: tenure, dtype: float64

**3.Transforming Categorical Variables**

**Note:** We are NOT using dummies here to minimize the explosion of columns because of the distance methods we are using.

In [88]:
# Now we need to transform the features of the feature store.
def encode_categorical_features(train_df,prod_df):
    # Get a list of all categorical columns
    cat_columns = train_df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Encode each categorical column
    for col in cat_columns:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col])
        prod_df[col]= le.transform(prod_df[col])
    return train_df, prod_df

In [89]:
#excluding the customer ID so it doesn't get encoded
train_label_data=train_feat[train_feat.columns.difference(['Customer ID','Month','Month of Joining'])]
prod_label_data=prod_feat[prod_feat.columns.difference(['Customer ID','Month','Month of Joining'])]
train_feat_enc, prod_feat_enc = encode_categorical_features(train_label_data,prod_label_data)

/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/1391466617.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[col] = le.fit_transform(train_df[col])
/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/1391466617.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prod_df[col]= le.transform(prod_df[col])
/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/1391466617.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [90]:
##bringing back the customer ids keys
train_feat_enc['Customer ID'] = train_feat['Customer ID'] #bringing back the customer id
train_feat_enc['Month'] = train_feat['Month'] #bringing back the Month
train_feat_enc['Month of Joining'] = train_feat['Month of Joining'] #bringing back the Month of joining

prod_feat_enc['Customer ID'] = prod_feat['Customer ID'] #bringing back the customer id
prod_feat_enc['Month'] = prod_feat['Month'] #bringing back the Month
prod_feat_enc['Month of Joining'] = prod_feat['Month of Joining'] #bringing back the Month of joining


/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/289144358.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_feat_enc['Customer ID'] = train_feat['Customer ID'] #bringing back the customer id
/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/289144358.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_feat_enc['Month'] = train_feat['Month'] #bringing back the Month
/var/folders/dy/p7g7n6253j76sj1k7y99nbv80000gn/T/ipykernel_67607/289144358.py:6: SettingWithCopyWa

**4. Taking a look at the final list of variables**


In [91]:
train_feat_enc.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Age,159109.0,36.634975,12.179496,19.000000,28.00,34.00,43.000000,80.000000
Device Protection Plan,159109.0,0.498514,0.499999,0.000000,0.00,0.00,1.000000,1.000000
Gender,159109.0,0.782690,0.842949,0.000000,0.00,1.00,1.000000,3.000000
Internet Service,159109.0,0.609670,0.487826,0.000000,0.00,1.00,1.000000,1.000000
Internet Type,159109.0,2.031400,1.142408,0.000000,1.00,2.00,3.000000,3.000000
Married,159109.0,1.005041,0.921703,0.000000,0.00,1.00,2.000000,2.000000
Multiple Lines,159109.0,0.577089,0.621380,0.000000,0.00,1.00,1.000000,2.000000
Number of Dependents,159109.0,1.162522,2.248864,0.000000,0.00,0.00,1.000000,9.000000
Number of Referrals,159109.0,4.338764,3.768069,0.000000,0.00,4.00,8.000000,11.000000
Online Backup,159109.0,0.331169,0.470635,0.000000,0.00,0.00,1.000000,1.000000


In [ ]:
train = pd.merge(train_feat_enc,train_id[['Customer ID','Month','Month of Joining','Churn Value','offer']],how = 'inner',on=['Customer ID','Month','Month of Joining'])
production = pd.merge(prod_feat_enc,prod_id[['Customer ID','Month','Month of Joining','Churn Value','offer']],how = 'inner',on=['Customer ID','Month','Month of Joining'])

In [93]:
## This help us check if we did not duplicate anything in the columns

print(len(train))
print(len(production))

159109
494644


## **Model Building and Testing**